# Semana 12: Autovalores y Autovectores - Las Direcciones Clave
## Notebook de Ejercicios (NB2) - PCA en MNIST

### Propósito de la sesión
Aplicar los conceptos de **autovalores y autovectores** para implementar **PCA (Análisis de Componentes Principales)** desde cero en el dataset MNIST. Visualizaremos los dígitos proyectados en 2D, compararemos nuestra implementación con `sklearn.decomposition.PCA` y observaremos cómo los primeros autovectores (componentes principales) se asemejan a "prototipos" de dígitos.

### Objetivos de aprendizaje
*   Cargar y preprocesar el dataset **MNIST**.
*   Implementar **PCA desde cero** usando autovalores y autovectores de la matriz de covarianza.
*   Proyectar los dígitos a 2D y visualizarlos, coloreando por clase.
*   Comparar nuestra implementación con `sklearn.decomposition.PCA`.
*   Visualizar los **autovectores** (componentes principales) como imágenes y comprender que representan "prototipos" o "modos de variación".

## Configuración Inicial

Importamos las librerías necesarias: numpy, matplotlib, sklearn para datasets y PCA, y torchvision para MNIST.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 10)

# Fijamos semilla para reproducibilidad
np.random.seed(42)

print("🎯 Librerías importadas correctamente.")

---
## 1. Carga del Dataset MNIST

Cargamos MNIST usando `fetch_openml` de sklearn. MNIST contiene 70,000 imágenes de 28x28 píxeles (784 dimensiones). Para acelerar los cálculos, tomaremos un subconjunto.

In [ ]:
# Cargamos MNIST
print("Cargando MNIST...")
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False, parser='auto')

# Convertimos etiquetas a enteros
y = y.astype(int)

# Tomamos un subconjunto para que sea más rápido (5000 imágenes)
n_samples = 5000
indices = np.random.choice(len(X), n_samples, replace=False)
X = X[indices]
y = y[indices]

print(f"Shape de X: {X.shape} ({n_samples} muestras, 784 características)")
print(f"Shape de y: {y.shape}")
print(f"Clases presentes: {np.unique(y)}")

In [ ]:
# Visualizamos algunas imágenes
def plot_mnist_samples(X, y, num_samples=10):
    plt.figure(figsize=(15, 6))
    for i in range(num_samples):
        plt.subplot(2, 5, i+1)
        plt.imshow(X[i].reshape(28, 28), cmap='gray')
        plt.title(f'Dígito: {y[i]}')
        plt.axis('off')
    plt.suptitle('Muestras de MNIST', fontsize=16)
    plt.tight_layout()
    plt.show()

plot_mnist_samples(X, y)

---
## 2. Implementación de PCA desde cero

Pasos del PCA:
1. Centrar los datos (restar la media por característica).
2. Calcular la matriz de covarianza: $\Sigma = \frac{1}{n-1} X_c^T X_c$.
3. Calcular autovalores y autovectores de $\Sigma$.
4. Ordenar autovalores y autovectores de mayor a menor.
5. Seleccionar los primeros $k$ autovectores (componentes principales).
6. Proyectar los datos: $X_{proj} = X_c W$, donde $W$ contiene los $k$ autovectores.

In [ ]:
class PCAFromScratch:
    def __init__(self, n_components):
        self.n_components = n_components
        self.components_ = None
        self.mean_ = None
        self.explained_variance_ = None
        self.explained_variance_ratio_ = None
    
    def fit(self, X):
        # 1. Centrar los datos
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_
        
        # 2. Calcular matriz de covarianza
        # Usamos la fórmula (X_centered.T @ X_centered) / (n-1)
        n = X.shape[0]
        cov_matrix = (X_centered.T @ X_centered) / (n - 1)
        
        # 3. Calcular autovalores y autovectores
        eigvals, eigvecs = np.linalg.eig(cov_matrix)
        
        # 4. Ordenar de mayor a menor
        idx = np.argsort(eigvals)[::-1]
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]
        
        # 5. Guardar los primeros n_components
        self.components_ = eigvecs[:, :self.n_components]
        self.explained_variance_ = eigvals[:self.n_components]
        self.explained_variance_ratio_ = eigvals[:self.n_components] / np.sum(eigvals)
        
        return self
    
    def transform(self, X):
        X_centered = X - self.mean_
        return X_centered @ self.components_
    
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

# Aplicamos PCA desde cero con 2 componentes
pca_scratch = PCAFromScratch(n_components=2)
X_pca_scratch = pca_scratch.fit_transform(X)

print(f"Shape después de PCA (scratch): {X_pca_scratch.shape}")
print(f"Varianza explicada (primeros 2): {pca_scratch.explained_variance_}")
print(f"Proporción de varianza explicada: {pca_scratch.explained_variance_ratio_}")
print(f"Varianza total explicada: {np.sum(pca_scratch.explained_variance_ratio_):.4f}")

---
## Actividad 1: Visualizar los dígitos proyectados a 2D

In [ ]:
plt.figure(figsize=(14, 10))
scatter = plt.scatter(X_pca_scratch[:, 0], X_pca_scratch[:, 1], c=y, cmap='tab10', alpha=0.6, s=10)
plt.colorbar(scatter, ticks=range(10), label='Dígito')
plt.xlabel('Primera Componente Principal')
plt.ylabel('Segunda Componente Principal')
plt.title('Proyección PCA 2D de MNIST (implementación desde cero)')
plt.grid(True)
plt.show()

print("📌 Observamos cierta separación de los dígitos, aunque con superposiciones.")
print("📌 Las dos primeras componentes retienen solo una pequeña fracción de la varianza total.")

---
## Actividad 2: Comparar con sklearn.decomposition.PCA

In [ ]:
# PCA con sklearn
pca_sklearn = SklearnPCA(n_components=2)
X_pca_sklearn = pca_sklearn.fit_transform(X)

print("🔷 sklearn PCA:")
print(f"Varianza explicada: {pca_sklearn.explained_variance_}")
print(f"Proporción de varianza: {pca_sklearn.explained_variance_ratio_}")

# Comparación de proyecciones (pueden diferir en signo, pero las direcciones son las mismas)
# Corregimos posible cambio de signo
from sklearn.utils.extmath import svd_flip
# Ajustamos signos para comparación
for i in range(2):
    if np.dot(pca_scratch.components_[:, i], pca_sklearn.components_[i]) < 0:
        pca_scratch.components_[:, i] *= -1
        X_pca_scratch[:, i] *= -1

# Comparación de componentes
print("\n📊 Comparación de componentes principales (primeros 5 elementos):")
print("Componente 1 (scratch):", pca_scratch.components_[:5, 0])
print("Componente 1 (sklearn):", pca_sklearn.components_[0, :5])
print("\nComponente 2 (scratch):", pca_scratch.components_[:5, 1])
print("Componente 2 (sklearn):", pca_sklearn.components_[1, :5])

# Error máximo entre proyecciones
diff = np.abs(X_pca_scratch - X_pca_sklearn)
print(f"\nDiferencia máxima en proyecciones: {np.max(diff):.2e}")

# Visualización lado a lado
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.scatter(X_pca_scratch[:, 0], X_pca_scratch[:, 1], c=y, cmap='tab10', alpha=0.6, s=5)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA desde cero')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(X_pca_sklearn[:, 0], X_pca_sklearn[:, 1], c=y, cmap='tab10', alpha=0.6, s=5)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA sklearn')
plt.grid(True)

plt.suptitle('Comparación de proyecciones PCA')
plt.tight_layout()
plt.show()

---
## Actividad 3: Visualizar los autovectores (componentes principales) como imágenes

Los primeros autovectores (componentes principales) pueden visualizarse como imágenes de 28x28. Estos representan los "prototipos" o "modos de variación" principales en los datos.

In [ ]:
# Recalculamos PCA con más componentes para visualizar (usamos sklearn por rapidez)
pca_viz = SklearnPCA(n_components=10)
pca_viz.fit(X)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    component = pca_viz.components_[i].reshape(28, 28)
    ax.imshow(component, cmap='RdBu', interpolation='nearest')
    ax.set_title(f'Componente {i+1}')
    ax.axis('off')

plt.suptitle('Primeros 10 autovectores (componentes principales) de MNIST', fontsize=16)
plt.tight_layout()
plt.show()

### 3.1. Interpretación

Observamos que:
*   La **primera componente** captura la variación más global (por ejemplo, la intensidad general o el contraste entre el centro y los bordes).
*   Las componentes siguientes capturan detalles más finos: trazos curvos, diferencias entre dígitos con bucles, etc.
*   Estos autovectores no son dígitos reconocibles por sí mismos, sino **direcciones en el espacio de imágenes** que, al sumarse o restarse de la imagen media, producen variaciones típicas en los datos.

Por ejemplo, la imagen promedio de todos los dígitos más un múltiplo de la primera componente podría aclarar o oscurecer ciertas regiones.

In [ ]:
# Visualización de la imagen media y efectos de las componentes
mean_image = pca_viz.mean_.reshape(28, 28)

plt.figure(figsize=(12, 4))
plt.subplot(1, 4, 1)
plt.imshow(mean_image, cmap='gray')
plt.title('Imagen media')
plt.axis('off')

for i in range(1, 4):
    # Media + 3 * componente (para exagerar)
    comp = pca_viz.components_[i-1].reshape(28, 28)
    img_plus = mean_image + 3 * comp
    img_plus = np.clip(img_plus, 0, 255)  # para visualización
    plt.subplot(1, 4, i+1)
    plt.imshow(img_plus, cmap='gray')
    plt.title(f'Media + 3*C{i}')
    plt.axis('off')

plt.suptitle('Efecto de añadir componentes principales a la imagen media')
plt.tight_layout()
plt.show()

---
## Actividad 4: Varianza explicada y selección de componentes

Analizamos cuántas componentes se necesitan para retener diferentes porcentajes de varianza.

In [ ]:
# Calculamos PCA con todas las componentes (hasta el mínimo)
pca_full = SklearnPCA()
pca_full.fit(X)

# Varianza explicada acumulada
cumsum = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumsum)+1), cumsum, 'b-', linewidth=2)
plt.axhline(y=0.95, color='r', linestyle='--', label='95% varianza')
plt.axhline(y=0.90, color='g', linestyle='--', label='90% varianza')
plt.xlabel('Número de componentes')
plt.ylabel('Varianza explicada acumulada')
plt.title('Varianza explicada vs número de componentes en MNIST')
plt.legend()
plt.grid(True)
plt.xlim(0, 200)
plt.show()

# Encontrar número de componentes para 95%
n_95 = np.argmax(cumsum >= 0.95) + 1
print(f"Para explicar el 95% de la varianza se necesitan {n_95} componentes.")
print(f"Esto reduce la dimensionalidad de 784 a {n_95}, ¡una gran compresión!")

---
## Ejercicios para el Estudiante

1.  **Normalización:** ¿Qué diferencia hay si estandarizamos los datos (normalizar por desviación estándar) antes de PCA? Prueba con `StandardScaler` y compara las componentes.

2.  **Número de componentes:** Proyecta MNIST a 3D y visualiza (puedes usar proyección 3D con matplotlib). ¿Se observan mejor las separaciones?

3.  **Reconstrucción:** Elige una imagen de prueba, proyéctala a un número reducido de componentes (ej. 50) y reconstruye la imagen. Compara visualmente la original y la reconstruida. ¿Qué detalles se pierden?

4.  **PCA en otros datasets:** Aplica PCA a Fashion-MNIST o a CIFAR-10 (convertido a grises). ¿Cómo son los autovectores en esos casos?

5.  **Reflexión:** En el contexto de modelos generativos (VAEs), el espacio latente se aprende, no se obtiene por PCA. ¿Qué ventajas y desventajas tiene usar autovectores (PCA) frente a un espacio latente aprendido?

---
## Conclusiones de la Sesión

*   Implementamos **PCA desde cero** usando autovalores y autovectores de la matriz de covarianza, confirmando nuestra comprensión teórica.
*   Proyectamos los dígitos MNIST a 2D, observando que las dos primeras componentes ofrecen una visualización útil aunque con superposiciones.
*   Comparamos nuestra implementación con `sklearn.decomposition.PCA`, verificando que producen resultados prácticamente idénticos (salvo signo).
*   Visualizamos los **autovectores** (componentes principales) como imágenes, notando que representan los principales modos de variación en los datos, no dígitos reconocibles.
*   Analizamos la **varianza explicada**, mostrando que con relativamente pocas componentes se puede retener gran parte de la información.
*   Conectamos estos conceptos con aplicaciones reales: reducción de dimensionalidad, visualización, compresión y análisis de datos.

¡Ahora sabes cómo los autovalores y autovectores son la base del PCA, una herramienta fundamental en machine learning!